# Customer Churn Analysis
**Exploratory Data Analysis on Telco Customer Churn Dataset**

Goal: Understand why customers churn and identify key drivers using Python and data visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')

print('Libraries loaded successfully')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/churn.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 2. Data Types & Missing Values

In [ ]:
print('Data types:\n')
print(df.dtypes)

print('\nMissing values per column:')
print(df.isnull().sum())

print('\nChurn distribution:')
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

## 3. Data Cleaning

In [ ]:
# Fix TotalCharges: stored as string, convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('NaN in TotalCharges after conversion:', df['TotalCharges'].isnull().sum())

# Drop rows where TotalCharges is NaN (new customers with 0 tenure)
df.dropna(subset=['TotalCharges'], inplace=True)

# Convert Churn to binary
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Drop customerID
df.drop(columns=['customerID'], inplace=True)

print('Cleaned dataset shape:', df.shape)
print('Overall churn rate:', round(df['Churn'].mean() * 100, 2), '%')

## 4. Churn Distribution

In [ ]:
fig, ax = plt.subplots()
counts = df['Churn'].value_counts()
bars = ax.bar(['Retained', 'Churned'], counts.values, color=['#2E75B6', '#E05C5C'], width=0.5)

for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontweight='bold')

ax.set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Customers')
ax.set_ylim(0, counts.max() + 400)
plt.tight_layout()
plt.savefig('../data/churn_distribution.png', bbox_inches='tight')
plt.show()
print(f'Churn rate: {df["Churn"].mean()*100:.1f}% of customers churned')

## 5. Churn Rate by Contract Type

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].mean().mul(100).round(1).reset_index()
contract_churn.columns = ['Contract Type', 'Churn Rate (%)']

fig, ax = plt.subplots()
bars = ax.bar(contract_churn['Contract Type'], contract_churn['Churn Rate (%)'],
              color=['#E05C5C', '#F0A500', '#2E75B6'], width=0.5)

for bar, val in zip(bars, contract_churn['Churn Rate (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}%', ha='center', fontweight='bold')

ax.set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 60)
plt.tight_layout()
plt.savefig('../data/contract_churn.png', bbox_inches='tight')
plt.show()

## 6. Tenure & Monthly Charges vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[df['Churn']==0]['tenure'], bins=30, alpha=0.7, color='#2E75B6', label='Retained')
axes[0].hist(df[df['Churn']==1]['tenure'], bins=30, alpha=0.7, color='#E05C5C', label='Churned')
axes[0].set_title('Tenure Distribution: Churned vs Retained')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(df[df['Churn']==0]['MonthlyCharges'], bins=30, alpha=0.7, color='#2E75B6', label='Retained')
axes[1].hist(df[df['Churn']==1]['MonthlyCharges'], bins=30, alpha=0.7, color='#E05C5C', label='Churned')
axes[1].set_title('Monthly Charges: Churned vs Retained')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/tenure_charges.png', bbox_inches='tight')
plt.show()

## 7. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(numeric_df.corr(), dtype=bool))
sns.heatmap(numeric_df.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.5, square=True)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 8. Key Findings Summary

In [ ]:
print('=' * 55)
print('   CUSTOMER CHURN ANALYSIS — KEY FINDINGS')
print('=' * 55)

total = len(df)
churned = df['Churn'].sum()
churn_rate = churned / total * 100

print(f'\nDataset: {total:,} customers | {churned:,} churned ({churn_rate:.1f}%)')

print('\nTop Churn Drivers:')
print(f'  Month-to-month contracts: {df[df["Contract"]=="Month-to-month"]["Churn"].mean()*100:.1f}% churn rate')
print(f'  Two-year contracts:       {df[df["Contract"]=="Two year"]["Churn"].mean()*100:.1f}% churn rate')

early = df[df['tenure'] <= 12]['Churn'].mean() * 100
late  = df[df['tenure'] > 24]['Churn'].mean() * 100
print(f'  Customers in first 12 months: {early:.1f}% churn rate')
print(f'  Customers beyond 24 months:   {late:.1f}% churn rate')

high_bill = df[df['MonthlyCharges'] > 70]['Churn'].mean() * 100
low_bill  = df[df['MonthlyCharges'] <= 70]['Churn'].mean() * 100
print(f'  High monthly charges (>$70):  {high_bill:.1f}% churn rate')
print(f'  Lower monthly charges (<=70): {low_bill:.1f}% churn rate')

print('\nBusiness Recommendations:')
print('  1. Offer discounts to convert month-to-month users to annual plans')
print('  2. Prioritise retention campaigns in the first 12 months')
print('  3. Review pricing strategy for high-charge customers')
print('=' * 55)